# 05 参数优化与过拟合检验

1. 网格搜索 — 全量枚举参数组合
2. 遗传算法 — 大参数空间高效搜索
3. Walk-Forward Analysis — 样本外验证，防止过拟合

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from futuresquant.data.loader import FuturesDataLoader
from futuresquant.backtest.engine import BacktestConfig
from futuresquant.optimize import GridSearchOptimizer, GeneticOptimizer, WalkForwardOptimizer
from futuresquant.strategy.examples import MACrossStrategy, ATRBreakoutStrategy

DATA_DIR = r'I:\stock\FuturesQuant\raw_data\1min_FU'
CACHE_DIR = r'I:\stock\FuturesQuant\cache'
SYMBOL = 'FU2210'
CONFIG = BacktestConfig(symbol=SYMBOL, initial_capital=1_000_000)

loader = FuturesDataLoader(DATA_DIR, cache_dir=CACHE_DIR)
klines_full = loader.load(SYMBOL, start='2022-01-01', end='2022-10-31')

# IS / OOS 手动分割（用于对比演示）
split_date = '2022-07-01'
klines_is  = klines_full[klines_full.index < split_date]
klines_oos = klines_full[klines_full.index >= split_date]
print(f'IS: {len(klines_is):,} bars   OOS: {len(klines_oos):,} bars')

## 1. 网格搜索

In [ ]:
factory = lambda fast, slow: MACrossStrategy(fast=fast, slow=slow)
param_grid = {
    'fast': [3, 5, 10, 15, 20],
    'slow': [20, 30, 40, 60, 80],
}
constraints = [lambda p: p['fast'] < p['slow']]

grid_opt = GridSearchOptimizer(
    factory, param_grid, CONFIG,
    metric='sharpe', constraints=constraints
)
grid_result = grid_opt.run(klines_is)

print(f'最优参数: {grid_result.best_params}  Sharpe={grid_result.best_score:.4f}')
grid_result.summary(top_n=8)

In [ ]:
# Sharpe 热图
heatmap = grid_result.heatmap_data('fast', 'slow')
fig = px.imshow(
    heatmap, text_auto='.3f', color_continuous_scale='RdYlGn',
    title='网格搜索 Sharpe 热图（IS 期）',
    labels={'x': 'fast window', 'y': 'slow window', 'color': 'Sharpe'}
)
fig.update_layout(height=380)
fig.show()

## 2. 遗传算法（大参数空间）

In [ ]:
# 扩大搜索空间，网格搜索会有 ~100 组合，遗传算法只跑 population×generation 次
large_grid = {
    'fast':  list(range(3, 25, 2)),   # 3,5,7,...,23  → 11 个
    'slow':  list(range(20, 100, 5)), # 20,25,...,95  → 16 个
}
# 全量 = 176 组合，遗传算法只评估 pop×gen ≈ 50×10=500 次（含缓存命中会更少）

ga_opt = GeneticOptimizer(
    factory, large_grid, CONFIG,
    metric='sharpe', constraints=constraints,
    population_size=20, n_generations=10,
    mutation_rate=0.2, seed=42,
)
ga_result = ga_opt.run(klines_is)

print(f'遗传算法最优参数: {ga_result.best_params}  Sharpe={ga_result.best_score:.4f}')
print(f'实际评估次数（去重）: {len(ga_result.records)}')
ga_result.summary(top_n=5)

In [ ]:
# IS vs OOS：验证最优参数在样本外是否仍然有效
from futuresquant.backtest.engine import BacktestEngine

best = ga_result.best_params
is_res  = BacktestEngine(factory(**best), CONFIG).run(klines_is)
oos_res = BacktestEngine(factory(**best), CONFIG).run(klines_oos)

print(f"参数 {best}")
print(f"  IS  Sharpe = {is_res.metrics['sharpe']:.4f}   MaxDD = {is_res.metrics['max_drawdown']:.2%}")
print(f"  OOS Sharpe = {oos_res.metrics['sharpe']:.4f}   MaxDD = {oos_res.metrics['max_drawdown']:.2%}")
ratio = oos_res.metrics['sharpe'] / is_res.metrics['sharpe'] if is_res.metrics['sharpe'] != 0 else float('nan')
print(f"  OOS/IS = {ratio:.2f}  ({'✓ 可接受' if ratio > 0.5 else '⚠ 过拟合风险'})")

## 3. Walk-Forward Analysis（防过拟合核心工具）

In [ ]:
wf_opt = WalkForwardOptimizer(
    factory,
    param_grid={'fast': [3, 5, 10], 'slow': [20, 30, 40]},
    config=CONFIG,
    metric='sharpe',
    constraints=constraints,
    n_splits=4,
    is_ratio=0.65,
    optimizer='grid',
)
wf_result = wf_opt.run(klines_full)

print(f'\nOOS 综合绩效:')
for k, v in wf_result.oos_metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
print(f'\nIS/OOS 衰减比: {wf_result.degradation_ratio:.3f}', end='  ')
if wf_result.degradation_ratio > 0.5:
    print('✓ 参数稳健')
elif wf_result.degradation_ratio > 0:
    print('⚠ 一定程度过拟合')
else:
    print('✗ 严重过拟合')

In [ ]:
# WFA 汇总表
wf_result.summary()

In [ ]:
# IS vs OOS Sharpe 对比柱状图
df_sum = wf_result.summary()
fig = go.Figure()
fig.add_trace(go.Bar(x=df_sum['窗口'].astype(str), y=df_sum['IS sharpe'],
                      name='IS Sharpe', marker_color='steelblue'))
fig.add_trace(go.Bar(x=df_sum['窗口'].astype(str), y=df_sum['OOS sharpe'],
                      name='OOS Sharpe', marker_color='#2ca02c'))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(title='各窗口 IS vs OOS Sharpe', barmode='group',
                   xaxis_title='窗口编号', yaxis_title='Sharpe', height=380)
fig.show()

In [ ]:
# 参数稳定性：各窗口最优参数是否一致
stab = wf_result.param_stability()
print('各窗口最优参数:')
print(stab.to_string(index=False))

# 如果 slow 每窗口都相同 → 参数稳定，信号来自结构性规律而非噪音
for col in [c for c in stab.columns if c != '窗口']:
    uniq = stab[col].nunique()
    print(f'{col}: {uniq} 种取值 ({'稳定' if uniq == 1 else '变化'})')

In [ ]:
# OOS 拼接权益曲线（WFA 真实绩效）vs 全样本最优权益
oos_eq = wf_result.oos_equity.resample('1D').last().dropna()

# 全样本最优（存在前视偏差）
full_opt_res = BacktestEngine(
    factory(**grid_result.best_params), CONFIG
).run(klines_full)
full_eq = full_opt_res.account.equity_curve().resample('1D').last().dropna()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=oos_eq.index, y=oos_eq / CONFIG.initial_capital,
    name='WFA OOS（真实估计）', line=dict(color='#2ca02c', width=2)
))
fig.add_trace(go.Scatter(
    x=full_eq.index, y=full_eq / CONFIG.initial_capital,
    name='全样本最优（有前视偏差）', line=dict(color='#d62728', dash='dash')
))
fig.add_hline(y=1, line_dash='dot', line_color='gray')
fig.update_layout(
    title='WFA OOS 权益 vs 全样本最优（差距 = 过拟合程度）',
    yaxis_title='归一化权益', height=420, hovermode='x unified'
)
fig.show()